In [3]:
from astropy.utils import iers
iers.conf.auto_download = False

import pathlib
import numpy as np
import pandas as pd

here = pathlib.Path.cwd()
root = here.parent if here.name.lower() == "notebooks" else here
assert root.name == "Sunspot detection with ML", f"Wrong root: {root}"

DATA      = root / "Data"
RAW_HMI   = DATA / "Raw" / "HMI"
REF_DEB   = DATA / "Reference" / "Debrecen"
REF_NOAA  = DATA / "Reference" / "NOAA"
PROCESSED = DATA / "Processed"

for d in [RAW_HMI, REF_DEB, REF_NOAA, PROCESSED, root / "Src"]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", root)

Project root: /Users/vinuradesilva/Documents/Projects/Sunspot detection with ML


In [4]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"   # must be set BEFORE importing torch
import torch

print("torch:", torch.__version__)
print("MPS built    :", torch.backends.mps.is_built())
print("MPS available:", torch.backends.mps.is_available())

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
x = torch.randn(2000, 2000, device=device)
y = x @ x
print("Computed on:", y.device, "| dtype:", y.dtype)

torch: 2.14.1
MPS built    : True
MPS available: True
Computed on: mps:0 | dtype: torch.float32


In [5]:
from collections import Counter

for name in ["HMIDD2014.txt", "fdHMIDD2014.txt"]:
    f = REF_DEB / name
    with open(f) as fh:
        lines = fh.readlines()
    print(f"\n=== {name}: {f.stat().st_size/1e6:.1f} MB, {len(lines)} lines")
    print("Row types:", Counter(l[0] for l in lines if l.strip()))
    for l in lines[:8]:
        print(repr(l))


=== HMIDD2014.txt: 283.0 MB, 2111146 lines
Row types: Counter({'s': 1961470, 'g': 141014, 'h': 8662})
'h 2014 01 01 00 00 24.20 SHMI          19   333.46  1302.79   542.98  1546.36 2456658.50028009    0.07  -3.00 189.47  999999  999999  999999\n'
'g 2014 01 01 00 00 24.20   11934       11    12.57    68.29    48.07   244.41   -15.35  271.70   82.23 254.87 0.9888    44.5    57.4\n'
's 2014 01 01 00 00 24.20   11934        1     6.03    29.48    28.56   146.87   -14.07  273.90   84.43 256.16 0.9948  -130.0    39.5\n'
's 2014 01 01 00 00 24.20   11934        2     4.67    13.59    13.89    40.44   -16.30  270.31   80.84 253.96 0.9867   423.9   276.4\n'
's 2014 01 01 00 00 24.20   11934        3     0.00     1.61     0.00     5.36   -16.81  271.32   81.85 253.45 0.9893  -111.0  -168.7\n'
's 2014 01 01 00 00 24.20   11934        4     0.00     1.10     0.00     3.27   -16.76  270.38   80.91 253.50 0.9869   363.0   339.9\n'
's 2014 01 01 00 00 24.20   11934        5     1.87     5.44     5.